<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
        except requests.RequestException as e:
            last = e                                        # network error: retry
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:         # our bug: raise now
                raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
            last = f'HTTP {r.status_code}'                  # their problem: retry
        if attempt < tries - 1:
            time.sleep(1 * (2 ** attempt))                  # 1s, 2s, ...
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
used_fallback = False

try:
    full = get_weather('2026-09-05', '2026-10-03')          # one request covering both
    wx = full[full['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    wx['source'] = 'open-meteo'
except Exception as e:
    print('API call failed, using fallback:', e)
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'

print('source:', wx['source'].unique().tolist())
wx

source: ['open-meteo']


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
wx = (
    wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    .assign(date=lambda d: pd.to_datetime(d['date']))
    [['date', 'temp_max', 'precip_mm', 'source']]
    .reset_index(drop=True)
)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
diff = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date']:%B %d} was the wetter game day with {wetter['precip_mm']:.1f} mm of rain, "
      f"{diff:.1f} mm more than {drier['date']:%B %d} ({drier['precip_mm']:.1f} mm). "
      f"Source: {wetter['source']}.")

October 03 was the wetter game day with 14.7 mm of rain, 12.2 mm more than September 05 (2.5 mm). Source: open-meteo.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
from datetime import datetime

print('Endpoint:    ', BASE)
print('Coordinates: ', '38.03, -78.51 (Charlottesville)')
print('Date range:  ', '2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)')
print('Pulled:      ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data source: ', 'FALLBACK SAMPLE (not real)' if used_fallback else 'real (Open-Meteo archive API)')

Endpoint:     https://archive-api.open-meteo.com/v1/archive
Coordinates:  38.03, -78.51 (Charlottesville)
Date range:   2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)
Pulled:       2026-10-08 17:52
Data source:  real (Open-Meteo archive API)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure, the merge onto 200 days of sales would be a left join that finds no weather rows, so every day gets NaN for rain. The chart then plots nothing, or shows rain as zero everywhere, and the relationship between rain and revenue shows no relationship. No errors, so the picture looks like a real finding: “rain doesn’t affect sales.” The conclusion rests on a network error, not on the weather. Nobody catches it because an empty frame from a failed call looks the same as a dry stretch of days, and the code runs cleanly to the end. Raising loudly, or recording the fallback in a source column, makes the failure visible before it reaches the findings.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
def city_weather(city, lat, lon):
    df = get_weather('2026-09-05', '2026-10-03', lat=lat, lon=lon)
    df = df[df['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    df = df.rename(columns={'time': 'date',
                            'temperature_2m_max': 'temp_max',
                            'precipitation_sum': 'precip_mm'})
    df['date'] = pd.to_datetime(df['date'])
    df['city'] = city
    df['source'] = 'open-meteo'
    return df[['city', 'date', 'temp_max', 'precip_mm', 'source']]

both = pd.concat([
    city_weather('Charlottesville', 38.03, -78.51),
    city_weather('Orlando', 28.54, -81.38),
], ignore_index=True)

assert len(both) == 4
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
